[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alg2338/inspect_ai/blob/colab/inspect_ai_tutorial_4_agents.ipynb)

# Tutorial 4: Evaluating LLM Agents on Mathematical Reasoning

Welcome to the fourth (and final) tutorial.

So far you have evaluated models as **passive responders** — the model reads a prompt,
produces an answer, and you score it. But many real-world AI systems are **agents**: they
observe, reason, act (e.g. call a tool), observe the result, and repeat. Evaluating agents
is harder because the model's behaviour is no longer a single forward pass — it is a
*multi-step trajectory* where mistakes compound and new failure modes (infinite loops,
tool misuse, hallucinated tool calls) appear.

In this tutorial you will build and evaluate a simple **ReAct agent** that solves
math problems by calling calculator and algebra tools. You will see first-hand how
scaffolding choices — prompts, tool sets, message limits, output formatting — affect
agent performance, and you will practice a basic dev/test workflow for iterating on
an agent without overfitting to your evaluation set. Think of it as a toy,
simplified version of a real elicitation pipeline like the
[METR Elicitation Protocol](https://evaluations.metr.org/elicitation-protocol/).

**What you'll learn:**

- Define custom tools for inspect_ai agents
- Build a ReAct agent and iteratively improve it on a dev set
- Develop intuition for how scaffolding choices affect agent performance
- (bonus) Try using the [Inspect Scout](https://meridianlabs-ai.github.io/inspect_scout/) tool

**By the end:** **You'll have a working agent evaluation pipeline and hands-on experience with the kind of iteration loop used in real-world agent evals.**

## 1. Setup

**Model choice.** We recommend picking a model that isn't too powerful — ideally one that occasionally
stumbles on arithmetic so you can observe the effect of giving it tools. The examples
below use `qwen2.5:3b`, but feel free to swap in any model you have access to (just make sure it supports tool calling). The
main goal is to see how well the model uses the tools it's given, so don't worry if
the tool-augmented score ends up lower than plain generation — that's a valid and
interesting finding, not a sign that something is broken. Conversely, if your model
solves everything perfectly even without tools, consider switching to a harder dataset
(e.g. the full MATH instead of MATH-500, or a competition-math set like AIME) — just
make sure to note this in your write-up.

> **Resource note:** Agent evaluations generate many more LLM calls than simple Q&A evals
> because each problem may involve multiple reasoning steps. All `eval()` calls in this
> notebook use a `limit` parameter to cap the number of samples processed. Adjust
> `EVAL_LIMIT` and `MAX_MESSAGES` if your machine is slow.

In [ ]:
%pip install -q inspect-ai openai datasets sympy scipy 
print("✅ Installed: inspect-ai, openai, datasets, sympy, scipy") 

In [ ]:
import re
import math
import random
from textwrap import dedent
from collections import defaultdict
from scipy.stats import norm

from inspect_ai import Task, eval
from inspect_ai.dataset import Sample, hf_dataset
from inspect_ai.agent import react
from inspect_ai.solver import generate, use_tools, system_message
from inspect_ai.scorer import (
    Score, Target, model_graded_qa, scorer, accuracy, stderr, match, mean
)
from inspect_ai.tool import tool

In [ ]:
# Configure model -- replace with what is available in your environment.
# E.g. 'ollama/qwen2.5:7b', 'openai/gpt-4o-mini', 'anthropic/claude-haiku-4-5'
# Must support tool calling (look for "tools" tag in Ollama).
# Provider choice and API keys work like in Tutorial 1: see its "How to pick a provider" table and .env.

MODEL = "ollama/qwen2.5:3b"

RANDOM_SEED = 42
EVAL_LIMIT = 30        # max samples per eval run (raise if your machine is fast)
MAX_MESSAGES = 20      # max back-and-forth messages per agent trajectory

A few helper functions for extracting and displaying results. You don't need to modify
these — just run the cell and move on.

In [ ]:
def get_acc(log):
    """Extract accuracy (or mean) from the first scorer in a log."""
    m = log.results.scores[0].metrics
    return (m.get("accuracy") or m.get("mean")).value


def _first_score(sample):
    """Get the first Score object from a sample, regardless of scorer name."""
    scores = sample.scores
    first_key = list(scores.keys())[0]
    val = scores[first_key]
    return val[0] if isinstance(val, list) else val


def print_results(label, log):
    """Pretty-print per-sample results from an eval log."""
    acc = get_acc(log)
    print(f"{'=' * 60}")
    print(f"  {label}   accuracy: {acc:.0%}")
    print(f"{'=' * 60}")
    for i, sample in enumerate(log.samples, 1):
        sc = _first_score(sample)
        msgs = len(sample.messages)
        expl = (sc.explanation or "")[:60]
        print(f"  [{sc.value}] #{i:2d}  msgs={msgs:2d}  "
              f"target={sample.target[:20]:>20s}  {expl}")
    print()

## 2. Tools and Agent Architecture

### Why agents?

A standard LLM evaluation looks like this: you hand the model a question, it produces
an answer, and a scorer checks whether the answer is correct. The model has *one shot*.

But many practical AI systems need to **interact with the world** — search the web, run
code, query a database, or call an API — before they can answer. These systems are
called **agents**. An agent follows a loop:

1. **Observe** the current state (the question, plus any tool outputs so far).
2. **Think** about what to do next.
3. **Act** by calling a tool (or submitting a final answer).
4. **Observe** the tool's result, then go back to step 2.

This loop continues until the agent decides it has enough information to submit a final
answer, or until a safety limit (maximum number of steps) is reached.

### Why not just give the model tools and let it figure things out?

Because *access* to tools is not enough. The model also needs:

- A **system prompt** that tells it what tools exist and how to use them
- A **loop structure** that feeds tool results back so the model can reason about them
- A **stopping criterion** so the model knows when and how to submit its final answer
- **Message limits** to prevent runaway loops that burn tokens without progress

This combination of prompt + loop + stopping logic is called **scaffolding**, and it
can make or break agent performance. Whether an agent actually helps depends on the
task, the tools, the prompt, and the model — scaffolding is not a guaranteed win, but
understanding it is essential for evaluating agent systems.

### Approaches to giving a model tools

> #### Notebook scope: one agent pattern, not all of Inspect
>
> Inspect can evaluate many kinds of agents: built-in ReAct agents, Deep Agents,
> software-engineering agents, fully custom agents, multi-agent systems, agents connected
> through external frameworks, and even human baselines. See the Inspect docs on
> [Using Agents](https://inspect.aisi.org.uk/agents.html) for the broader picture.
>
> In this tutorial, we focus on a simple ReAct-style setup, where we define tools
> ourselves and let the model use them through Inspect.

The simplest way to add tools in inspect_ai is the **`use_tools()` + `generate()`** pattern. `use_tools()` registers a list of tool functions so the model can call them, and `generate()` runs a regular model generation with those tools available. By default, Inspect can resolve tool calls in a basic loop: if the model calls a tool, Inspect executes it, appends the result to the context, and continues until the model stops calling tools or a limit is reached.
```python
solver = [
    system_message("You have access to calculator tools."),
    use_tools([add(), multiply()]),
    generate(),
]
```

This is easy to set up and often enough for simple tasks: the model answers the question and may use tools along the way.

### The ReAct pattern

**ReAct** (Reason + Act) is a more advanced agent pattern. It wraps tool use into an
explicit agent loop:



think → call a tool → observe the result → repeat


This makes tool use more structured than the lightweight `use_tools()` + `generate()`
pattern, and gives Inspect more control over how the model continues, submits answers,
and handles longer trajectories.

```
┌─────────────────────────────────────────────────────────┐
│                    ReAct Agent Loop                     │
│                                                         │
│   ┌──────────┐    ┌──────────┐    ┌──────────┐          │
│   │  THINK   │───>│   ACT    │───>│ OBSERVE  │──┐       │
│   │          │    │          │    │          │  │       │
│   │ "I need  │    │ call     │    │ tool     │  │       │
│   │  to add  │    │ add(a,b) │    │ returns  │  │       │
│   │  these"  │    │          │    │ "111915" │  │       │
│   └──────────┘    └──────────┘    └──────────┘  │       │
│        ^                                        │       │
│        └────────────────────────────────────────┘       │
│                                                         │
│   Loop continues until the agent calls submit()         │
│   or the message limit is reached.                      │
└─────────────────────────────────────────────────────────┘
```

In **inspect_ai**, the `react()` solver implements this pattern. It:
1. Gives the model an agent prompt and a set of tools
2. Lets the model reason and call tools step by step
3. Appends each tool result to the context
4. Repeats until the model calls `submit()` or a limit is reached

The `submit()` tool is special: it marks the agent's final answer. You do not define
`submit()` yourself; `react()` adds it for you.

### What `react()` adds compared to `[use_tools(), generate()]`

`[use_tools(), generate()]` is the lightweight pattern: give the model tools and run a
regular generation with tool calls enabled.

`react()` is a full agent scaffold. It gives you more control over how the model works
with tools:

* **`submit()` — explicit final answer**  
  The agent must call a special tool to submit its answer, instead of just ending with text.

* **`on_continue` — recovery if the model stops too early**  
  Controls what happens if the model stops calling tools or forgets to call `submit()`.

* **`attempts` — multiple submissions**  
  Allows the agent to submit an answer, receive scoring feedback, and try again.

* **`compaction` and `truncation` — long trajectories**  
  Help manage long agent runs when the conversation history gets large.

* **`name` / `description` — multi-agent use**  
  Makes the ReAct agent usable inside multi-agent systems, where other agents need to know
  what this agent is and when to delegate to it.

There are more configuration options as well. 

You can read more in the [Inspect documentation](https://inspect.aisi.org.uk/react-agent.html)

The original [ReAct paper](https://arxiv.org/abs/2210.03629) explains the idea of interleaving reasoning and tool use.

### Defining tools in inspect_ai

An agent needs *actions* it can take in the world. In inspect_ai, actions are
**tools** — Python functions the model can call during its reasoning loop.

The `@tool` decorator registers a function so that inspect_ai can:
1. **Describe** it to the model (via the docstring and type hints — these become the
   tool schema the model sees).
2. **Execute** it when the model emits a tool-call message and return the result.

The pattern is a factory function (decorated with `@tool`) that returns an `async`
inner function. The inner function's docstring and parameter annotations are what the
model actually sees, so clear names and descriptions directly affect agent performance.

Below we define four arithmetic tools ourselves. Notice how each one:
- Takes typed parameters with descriptive `Args:` docstrings
- Returns a **string** (a convenient choice for these simple tools)
- Wraps execution in `try/except` so the agent gets a readable error instead of a crash

> **Built-in tools:** inspect_ai also ships with ready-made tools for common agent tasks — you don't need to write these yourself:
> - `bash()` — run shell commands
> - `python()` — execute Python code
> - `web_search()` — search the web
> - `web_browser()` — full browser interaction
> - `text_editor()` — read and edit files
>
> For the full list see the [Tools section of the inspect_ai documentation](https://inspect.aisi.org.uk/tools.html).
> In this tutorial we write our own tools from scratch to understand the mechanics,
> but in practice you will often mix custom tools with these built-in ones.

In [ ]:
@tool
def add():
    async def execute(a: float, b: float) -> str:
        """Add two numbers.

        Args:
            a: First number.
            b: Second number.
        """
        try:
            return str(float(a) + float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def subtract():
    async def execute(a: float, b: float) -> str:
        """Subtract b from a.

        Args:
            a: Number to subtract from.
            b: Number to subtract.
        """
        try:
            return str(float(a) - float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def multiply():
    async def execute(a: float, b: float) -> str:
        """Multiply two numbers.

        Args:
            a: First number.
            b: Second number.
        """
        try:
            return str(float(a) * float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def divide():
    async def execute(a: float, b: float) -> str:
        """Divide a by b.

        Args:
            a: Dividend.
            b: Divisor (must not be zero).
        """
        try:
            b_val = float(b)
            if b_val == 0:
                return "Error: division by zero."
            return str(float(a) / b_val)
        except Exception as e:
            return f"Error: {e}"
    return execute

## Assignment 1: Create a `modular_arithmetic` tool

Cryptography and number theory problems often require modular arithmetic —
for example, computing $7^{1000} \mod 13$ as part of a larger proof.

Following the pattern above, implement a `modular_arithmetic` tool with a clear docstring. 

This tool will help you build a more capable mathematical agent, as both basic arithmetic and 
modular arithmetic operations are challenging for LLMs.

In [ ]:
@tool
def modular_arithmetic():
    async def execute(a: int, b: int) -> str:
        """Compute a mod b.
        
        YOUR DESCRIPTION
        
        """
        # YOUR CODE HERE
        raise NotImplementedError
    return execute

Now let's see the tool in action. We run a small eval so the model attempts to use your tool. Don't worry if the model gets the answer wrong — what matters here is that the tool itself is correctly defined.

In [ ]:
_test_samples_mod = [
    Sample(input="What is 1000000 mod 397? Reply with just the number.", target="354"),
    Sample(input="What is 100 mod 10? Reply with just the number.", target="0"),
]

_log_mod_test = eval(
    Task(
        dataset=_test_samples_mod,
        solver=react(
            prompt="You have a modular_arithmetic(a, b) tool. Use it, then submit the result.",
            tools=[modular_arithmetic()],
            attempts=1,
        ),
        scorer=match(numeric=True),
        message_limit=10,
    ),
    model=MODEL,
)[0]

print_results("modular_arithmetic tool test", _log_mod_test)

> **Checking tool usage.** Run **`inspect view`** in the terminal to see the full
> message trace for each sample — tool calls and their responses appear as explicit
> steps. Alternatively, inspect `log.samples[i].messages` directly: each tool
> call appears as a message with `role="assistant"` and a non-empty **`tool_calls`** field.

---

### 💭 Questions for reflection

1. Did the model actually *use* your tool, or did it answer without using it?
   Open the eval log in the inspect_ai viewer (`inspect view`) and check the trace —
   you should see explicit tool call steps between the initial question and the final answer.
2. If the model skipped the tool, adjust the prompt in the `react()` call above and
   re-run until the model uses it. What did you change?

**Your answer:**

In [ ]:
ARITH_TOOLS = [add(), subtract(), multiply(), divide(), modular_arithmetic()]

## 3. Toy Evaluation — Three Solver Architectures

Before touching a real benchmark, let's try out three different solver architectures
on a small set of hand-crafted problems. These 12 word problems use numbers large
enough that a model without tools might make arithmetic errors.

We will compare three solver architectures:

- **Plain generation** - reads the question, produces an answer in one shot;
  solver: `generate()`
- **Naive tool loop** - gets access to tools; `generate()` handles tool calls and
  continues generation by default; solver: `use_tools()` + `generate()`
- **ReAct agent** - explicit think-act-observe loop with a `submit()` action to stop;
  solver: `react()`
  
The goal here is to see how each architecture behaves — both in terms of accuracy
and how the solver actually runs — before moving to a real benchmark.

In [ ]:
TOY_SAMPLES = [
    Sample(
        input=(
            "A semiconductor factory produced 48,397 chips on Monday "
            "and 63,518 chips on Tuesday. How many chips were produced in total?"
        ),
        target="111915",
    ),
    Sample(
        input=(
            "A government reserve had 874,203 barrels of oil. "
            "After an emergency release, 295,867 barrels were distributed. "
            "How many barrels remain in the reserve?"
        ),
        target="578336",
    ),
    Sample(
        input=(
            "A logistics company ships 4,738 containers, each holding 2,659 units. "
            "How many units are shipped in total?"
        ),
        target="12598342",
    ),
    Sample(
        input=(
            "A national census counted 8,743,291 residents across 6,473 districts. "
            "If residents are distributed equally, how many full residents "
            "are assigned per district?"
        ),
        target="1350",
    ),
    Sample(
        input=(
            "A satellite completes a full orbit every 397 minutes. "
            "After exactly 1,000,000 minutes of operation, how many minutes "
            "have passed since the last complete orbit?"
        ),
        target="354",
    ),
    Sample(
        input=(
            "A hospital ordered 12,475 boxes of supplies at 387 dollars per box. "
            "They received a bulk discount of 843,750 dollars off the total. "
            "How much did the hospital pay after the discount?"
        ),
        target="3984075",
    ),
    Sample(
        input=(
            "A city has 14 times as many residents as municipal employees. "
            "If the total number of residents and employees together is 489,375, "
            "how many municipal employees does the city have?"
        ),
        target="32625",
    ),
    Sample(
        input=(
            "An airline flew 3,847 domestic flights and 2,964 international flights "
            "last month. Each flight used an average of 8,753 liters of fuel. "
            "How many liters of fuel were used in total?"
        ),
        target="59616683",
    ),
    Sample(
        input=(
            "A clock tower rings a bell every 1,873 seconds. "
            "After 10,000,000 seconds have elapsed since midnight, "
            "how many seconds ago did the bell last ring?"
        ),
        target="53",
    ),
    Sample(
        input=(
            "A farm harvested 247,839 kg of wheat and 184,672 kg of barley. "
            "The grain is loaded into trucks that carry exactly 4,750 kg each. "
            "How many full truckloads can be made from all the grain?"
        ),
        target="91",
    ),
    Sample(
        input=(
            "A global streaming platform has 1,847,293,847,291 seconds of video content. "
            "Given that a day has 86,400 seconds, how many full days of content "
            "does the platform have?"
        ),
        target="21380715",
    ),
    Sample(
        input=(
            "A country's economy grew by 3,847 dollars per citizen in a year. "
            "The country has 847,293,847 citizens. "
            "What was the total economic growth in dollars?"
        ),
        target="3259539429409",
    ),
]

## 3a. Approach 0: Plain generation (no tools)

The simplest baseline: give the model a system prompt and ask it to solve the problem
directly. The solver is just `generate()` — a single forward pass with no tool access.

We score with `match(numeric=True)`, which extracts the first number from the model's
response and compares it to the target.

> **Note:** Pay attention to match() defaults: they affect scoring. For example, location="end" is used by default.

In [ ]:
SIMPLE_PROMPT = dedent("""    You are a math solver. Read the problem carefully, compute the answer,
    and respond with the final numeric result.
""")

log_toy_gen = eval(
    Task(
        dataset=TOY_SAMPLES,
        solver=[system_message(SIMPLE_PROMPT), generate()],
        scorer=match(numeric=True), 
        # Note: `match()` has non-trivial defaults, such as location="end".
        # Make sure these defaults match the expected answer format.
    ),
    model=MODEL,
)[0]

print_results("Approach 0: generate() only", log_toy_gen)

---

### 💭 Questions for reflection

1. Did the model get everything right, or did it make arithmetic errors on the larger numbers?
2. If there were errors, what do you think caused them?

**Your answer:**

## 3b. Approach A: Naive tool loop

Now we give the model access to our arithmetic tools via `use_tools()`, followed by
`generate()`. With the default `tool_calls="loop"`, Inspect executes requested tools,
adds their results to the conversation, and asks the model to continue. This repeats
until the model responds without tool calls or a configured limit stops the run.
The model chooses which tools to request, including the option to use none.

> `use_tools()` plus `generate()` is a supported tool-use pattern. The ReAct approach
> in the next section adds instructions for the reasoning process and a `submit()`
> action for completing the task.

See [Tool Basics](https://inspect.aisi.org.uk/tools.html) and the
[generate reference](https://inspect.aisi.org.uk/reference/inspect_ai.solver.html#generate).

In [ ]:
NAIVE_LOOP_PROMPT = dedent("""    You are a math solver with access to calculator tools.
    Break each problem into arithmetic steps and call one tool per step.
""")

log_toy_naive = eval(
    Task(
        dataset=TOY_SAMPLES,
        solver=[
            system_message(NAIVE_LOOP_PROMPT),
            use_tools(ARITH_TOOLS),
            generate(),
        ],
        scorer=match(numeric=True),
    ),
    model=MODEL,
)[0]

print_results("Approach A: use_tools + generate (naive loop)", log_toy_naive)

---

### 💭 Questions for reflection

1. Did having access to tools improve results compared to the baseline?
2. Did the model actually use the tools, or did it ignore them?

**Your answer:**

## 3c. Approach B: ReAct agent

Now let's use the `react()` solver — the full ReAct loop described in the introduction.
Notice the prompt explicitly tells the model to use tools and call `submit()` at the end.

In [ ]:
REACT_PROMPT_V1 = dedent("""    You are a math solver with access to calculator tools.
    Break each problem into arithmetic steps and call one tool per step.
    Don't calculate anything without tools.
    After getting the final numeric result, call submit() with ONLY the number.
""")

log_toy_react = eval(
    Task(
        dataset=TOY_SAMPLES,
        solver=react(prompt=REACT_PROMPT_V1, tools=ARITH_TOOLS, attempts=1),
        scorer=match(numeric=True),
        message_limit=20,
    ),
    model=MODEL,
    max_connections=1,  # cap parallel calls; lower for slow local models
    timeout=120,        # seconds per request before giving up
)[0]

print_results("Approach B: react() with simple prompt", log_toy_react)

## Comparing the three approaches

Let's see the results side by side. Pay attention not just to accuracy but also to the
number of messages — more messages means more LLM calls, which means more cost and latency.

In [ ]:
TOY_LOGS = [log_toy_gen, log_toy_naive, log_toy_react]
TOY_LABELS = ["generate only", "naive tool loop", "react v1"]

print(f"{'Approach':<25s} {'Acc':>5s}  {'Avg msgs':>8s}  {'Max msgs':>8s}")
print("-" * 50)
for label, log in zip(TOY_LABELS, TOY_LOGS):
    acc = get_acc(log)
    msg_list = [len(s.messages) for s in log.samples]
    avg_m = sum(msg_list) / len(msg_list)
    max_m = max(msg_list)
    print(f"{label:<25s} {acc:>4.0%}   {avg_m:>7.1f}   {max_m:>7d}")

---

### 💭 Questions for reflection

1. Which approach performed best? Was it also the most expensive (most messages)?
2. Open the eval and compare the naive tool loop and the ReAct
   agent traces. Did the model actually use the tools in both cases, and how does the
   structure of the traces differ?

**Your answer:**

## 4. Adding a Symbolic Algebra Tool

Arithmetic tools help with computation, but many math problems require solving
equations — "find x such that 3x + 7 = 22". A small model can't do this reliably
without tools, but SymPy (a Python symbolic math library) can solve it exactly.

<details>
    <summary><b>Evals note on toolcalls freezing</b></summary>
    
Sometimes you need to run tools that can freeze. This is the case with the SymPy solver: by default, it can run for an arbitrarily long time if the problem is complex enough. One approach to fix this is to call the potentially infinite tool itself in a subprocess and terminate the subprocess if it exceeds its time limit. For example, like this:

```python
import asyncio, sys

from inspect_ai.tool import tool

@tool
def addition_timeout_tool():
    async def execute(a: str, b: str) -> str:
        """compute a + b, use timeout limiter
        
        timeout is 10 seconds"""
        TIMEOUT = 10
        python_exe = sys.executable
        core_code = r'''
import time
a = input()
b = input()
time.sleep(20)
print(str(float(a) + float(b)))
'''
        proc = await asyncio.create_subprocess_exec(
            python_exe, '-u', '-c', core_code,
            stdin=asyncio.subprocess.PIPE,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
        )
        try:
            # some arguments processing here - change it, if you change number/names of arguments
            stdout, stderr = await asyncio.wait_for(proc.communicate(input=str(a).encode() + b'\n' + str(b).encode() + b'\n'), timeout=TIMEOUT)
        except asyncio.TimeoutError:
            try: proc.kill()
            except: pass
            await proc.wait()
            return "Error: timeout, subprocess killed after 10 seconds."
        except Exception as e:
            try: proc.kill()
            except: pass
            await proc.wait()
            return f"Error: {e}"
        out = stdout.decode().strip() if stdout else ''
        err = stderr.decode().strip() if stderr else ''
        if err:
            return f"Error: {err}"
        elif out:
            return f"Result: {out}"
        else:
            return "Result: No output."
    return execute
```

To check it, run 
```
await addition_timeout_tool()("2", "4")
```

(it should return error)

Then delete sleep time in `core_code` and run again definition and call.

(it should return correct result)
</details>

<details>
    <summary><b>Why does my small local model fail in the 4th notebook?</b></summary>

Maybe it's because the tool calls fail. You can check your log in `inspect view` transcripts.

1. The model can hallucinate operands — for example, it can try to call something like `add(1, 2)` or `addition(a: "1", b: "2+4")`. The same problems may occur with other tools; the more complex the format is, the more problems you get (see `sympy_solve` with the most difficult format). 

2. Tool calls can be imitated in the message text instead of making "real" tool calls. You'll see it in the text of messages.

3. You can also exceed the `message_limit` barrier (`Message Limit Exceeded` label in transcript) or the model can abandon the instruction to use the `submit(...)` tool and try to print the correct result in plain text.
</details>

## Assignment 2: Create the `sympy_solve` tool

Implement a tool that takes an equation string (e.g. `"2*x + 5 = 21"` or `"3*x**2 - 12 = 0"`)
and returns the solutions using SymPy. The tool should:

1. Parse the equation — if it contains `=`, split into left and right sides and solve `left - right = 0`
2. If there's no `=`, treat the input as an expression equal to zero
3. Solve for the symbol `x`
4. Return the solutions as a string, or `"No solution found."` if empty
5. Handle errors gracefully

In [ ]:
@tool
def sympy_solve():
    async def execute(equation: str) -> str:
        '''
        YOUR DESCRIPTION
        '''
        # YOUR CODE HERE
        raise NotImplementedError
    return execute

In [ ]:
# Run the eval — the model should use your tool to solve both equations.
_log_sympy_test = eval(
    Task(
        dataset=[
            Sample(
                input="Solve for x: 2*x + 5 = 21. Reply with just the number.",
                target="8",
            ),
            Sample(
                input="Solve for x: 2*x**2 + 12*x + 18 = 0. Reply with just the number.",
                target="-3",
            ),
        ],
        solver=react(
            prompt="You have a sympy_solve(equation) tool. Use it to solve the equation, then submit the result.",
            tools=[sympy_solve()],
            attempts=1,
        ),
        scorer=match(numeric=True),
        message_limit=10,
    ),
    model=MODEL,
)[0]

print_results("sympy_solve tool test", _log_sympy_test)

---

### 💭 Questions for reflection

1. Did the model use the `sympy_solve` tool, or did it try to solve the equations
   in its head? (Check the message counts.)
2. If it didn't use the tool, try adjusting the prompt in the `react()` call.
   What wording helped?

**Your answer:**

In [ ]:
# Bundle all tools
ARITH_TOOLS = [add(), subtract(), multiply(), divide(), modular_arithmetic()]
ALL_TOOLS = ARITH_TOOLS + [sympy_solve()]

## 5. Loading the MATH-500 Benchmark

Now that we have a working agent with tools, let's evaluate it on a real benchmark.
**MATH-500** is a 500-question subset of the MATH dataset (Hendrycks et al., 2021),
covering competition-level math across seven subjects. It's available on Hugging Face.

Not all subjects benefit equally from our tools — Geometry and Counting & Probability
involve spatial reasoning and combinatorics that our calculator tools can't help with.
We'll focus on the four subjects where arithmetic (5 operations) and algebra tools (sympy solver)
are most relevant: Algebra, Intermediate Algebra, Number Theory, and Prealgebra. 
We use this toy domain because it allows you to easily validate the answers and reasoning yourself.

### Extracting answers from MATH solutions

MATH stores answers inside `\boxed{...}` in the solution string. We need a helper to
extract them:

In [ ]:
TOOL_SUBJECTS = [
    "Algebra", "Number Theory", "Prealgebra", "Intermediate Algebra",
]


def extract_boxed(solution):
    """Extract the content of the last \\boxed{...} in a MATH solution string."""
    idx = solution.rfind("\\boxed{")
    if idx == -1:
        return solution.strip()
    start = idx + len("\\boxed{")
    depth = 1
    i = start
    while i < len(solution) and depth > 0:
        if solution[i] == "{":
            depth += 1
        elif solution[i] == "}":
            depth -= 1
        i += 1
    return solution[start:i - 1].strip()


def record_to_sample(record):
    """Convert a MATH-500 record into an inspect_ai Sample."""
    target = record.get("answer") or extract_boxed(record["solution"])
    return Sample(
        input=record["problem"],
        target=target,
        metadata={
            "level": int(record["level"]),
            "subject": record["subject"],
        },
    )

In [ ]:
full_dataset = hf_dataset(
    path="HuggingFaceH4/MATH-500",
    split="test",
    sample_fields=record_to_sample,
    cached=True,
)

print(f"Total MATH-500: {len(full_dataset)} samples")

subject_counts = defaultdict(int)
for s in full_dataset:
    subject_counts[s.metadata["subject"]] += 1

print(f"\n{'Subject':<30s} {'Count':>5s}")
print("-" * 37)
for subj in sorted(subject_counts):
    marker = " <-- tool-friendly" if subj in TOOL_SUBJECTS else ""
    print(f"{subj:<30s} {subject_counts[subj]:>5d}{marker}")

## Dev/test split

A core principle in evaluation: **never tune on your test set.** Iterating on the same
data you use for final scoring inflates results and makes them meaningless. We split the
tool-friendly subset into a small **dev set** (10%) for prompt and scaffolding iteration,
and a larger **test set** (90%) reserved for final evaluation only.

This mirrors the elicitation workflow described in [METR's Guidelines for Capability Elicitation](https://evaluations.metr.org/elicitation-protocol/): iterate against the dev set until failures stabilize, then run the test set once.

> **Adjust to your hardware.** If even the dev set takes too long, reduce `split_point`
> or set a smaller `EVAL_LIMIT`. The point is rapid iteration — you can always increase
> the test set size later.

In [ ]:
tool_dataset = [s for s in full_dataset if s.metadata["subject"] in TOOL_SUBJECTS]
print(f"Tool-friendly subset: {len(tool_dataset)} samples")

random.seed(RANDOM_SEED)
random.shuffle(tool_dataset)

split_point = int(len(tool_dataset) * 0.1)
DEV_SET = tool_dataset[:split_point]
TEST_SET = tool_dataset[split_point:]

print(f"DEV_SET:  {len(DEV_SET)} samples")
print(f"TEST_SET: {len(TEST_SET)} samples")

## 6. Scoring Mathematical Answers

For the toy problems we used `match(numeric=True)`, which works when answers are plain
numbers. But MATH answers can be fractions (`3/7`), expressions (`2\sqrt{5}`), or
formatted in different equivalent ways — a simple string match will miss many correct answers.

We covered `model_graded_qa()` in notebook 3. Here we apply it to math: the key challenge
is writing a grading prompt that handles equivalent notations (e.g. `1/2` vs `0.5` vs `\frac{1}{2}`).
Note that in notebook 3 we deliberately hid the reference answer from the grading model — here
there's no reason to do that, so you can pass the correct answer directly.

> **Trade-off:** Model-graded scoring is slower and noisier, but catches equivalences that
> string matching misses. For production evals you might want a more capable model for
> grading than the one being tested — think about what makes sense for your setup.

## Assignment 3: Define the math scorer

Define a `math_scorer` using `model_graded_qa()`. Write a grading prompt that instructs
the model to judge mathematical equivalence and respond with **C** (correct) or **I** (incorrect),
and choose which model should do the grading. Think about what edge cases matter: different
notations, equivalent fractions, simplified vs unsimplified forms.

In [ ]:
SCORER_MODEL = # YOUR CODE HERE 

GRADING_INSTRUCTIONS = # YOUR CODE HERE

MATH_SCORER = # YOUR CODE HERE

In [ ]:
# Run the scorer on two toy samples where we know the answer.
_scorer_test_samples = [
    Sample(input="What is 1+1?", target="2"),
    Sample(input="What is 10/4?", target="5/2"),
]

_log_scorer_test = eval(
    Task(
        dataset=_scorer_test_samples,
        solver=[system_message("Answer the math question. Reply with just the answer."), generate()],
        scorer=MATH_SCORER,
    ),
    model=MODEL,
)[0]

print_results("Scorer sanity check", _log_scorer_test)
print("Check: does the scorer correctly mark equivalent answers as C?")

---

### 💭 Questions for reflection

1. Did your scorer correctly handle the fraction equivalence (`10/4` vs `5/2`)?
2. What failure modes can you imagine for model-graded scoring?

**Your answer:**

## 7. Dev-Set Iteration — Building and Improving Your Agent

This is the core of agent evaluation: run on the dev set, see where the agent struggles,
and systematically improve it. The iteration loop:

1. Run the current agent on the dev set
2. Inspect failures — *why* did the agent get these wrong?
3. Hypothesize an improvement (better prompt? more tools? output formatting?) — and check
   whether the answer was actually correct but the scorer marked it wrong
4. Implement the change and re-evaluate on the dev set
5. Repeat

## Assignment 4: Build and improve a ReAct agent

Your goal is to build a ReAct agent and iterate on it using the dev set.
Does adding tools and scaffolding help on this task, and by how much?
Start by running a basic ReAct agent on the dev set, then look at the failures in the logs and iterate.

**Step 1 — Run a baseline.** Write a system prompt and run `react()` with `ALL_TOOLS` on the dev set.

**Step 2 — Inspect failures.** Open the logs and look at what went wrong. Common things to consider:
- Is the model ignoring tools and is failing?
- Is the answer mathematically correct but formatted wrong (e.g. `0.5` instead of `1/2`)?
- Is the model getting lost in multi-step problems?

**Step 3 — Iterate.** Based on what you see, try to improve. Some directions:
- Make the system prompt more explicit about strategy or answer format
- Add tools that cover operations the model struggles with (e.g. a single `calculator` tool, `gcd`, `factorial` or something very different)
- Add a format-extraction step after the `react()` loop if formatting is the main issue
- Reconsider the scorer if correct answers are being marked wrong

For each configuration you try, store the result as a `(description, log)` tuple in `DEV_RUNS` at the bottom — this will let you compare all your attempts in one table. Try at least two configurations.

In [ ]:
MY_REACT_PROMPT = # YOUR CODE HERE

log_attempt_1 = eval(
    Task(
        dataset=DEV_SET,
        solver=react(
            prompt=MY_REACT_PROMPT,
            tools=ALL_TOOLS,
            attempts=1,
        ),
        scorer=MATH_SCORER,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
    max_connections=1,
    timeout=120,
)[0]

print_results("Attempt 1 (baseline)", log_attempt_1)
DEV_RUNS = [("Attempt 1 (baseline)", log_attempt_1)]

In [ ]:
# Look at the failures in the logs and decide what to change:
# prompt, tools, scorer, add a format-extraction step after react() or something else.
# Duplicate this cell for each new configuration. Give each log a new name.

# YOUR CODE HERE: define your updated prompt, tools, scorer, or solver

log_attempt_2 = eval(
    Task(
        dataset=DEV_SET,
        solver=# YOUR CODE HERE,
        scorer=# YOUR CODE HERE,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
    max_connections=1,
    timeout=120,
)[0]

description_2 = # YOUR CODE HERE
print_results(description_2, log_attempt_2)
DEV_RUNS.append((description_2, log_attempt_2))

In [ ]:
print(f"{'Configuration':<40s} {'Dev Accuracy':>12s}")
print("=" * 54)
for description, log in DEV_RUNS:
    acc = get_acc(log)
    print(f"{description:<40s} {acc:>12.0%}")

---

### 💭 Questions for reflection

1. What modifications did you try? Which had the biggest impact?
2. What was the best dev-set accuracy you achieved? What configuration produced it?
3. Did any change that you expected to help actually hurt? Why might that be?
4. Look at the individual failures. What are the most common error types?

**Your answer:**

## 8. Test-Set Evaluation

You've iterated on the dev set and found your best configuration. Now it's time for the
moment of truth: evaluating on the held-out test set.

> **Run this section only once**, with your best configuration. Re-running and picking
> the best result would be "test-set contamination" — the same as peeking at a test set
> in ML.

## Assignment 5: Run and analyze your best configuration on the test set

Run your best agent configuration on the held-out test set, report accuracy with a 95%
confidence interval, and break down performance by subject and difficulty level. Note
anything that stands out.

In [ ]:
def wilson_ci(n_correct, n_total, confidence_level=0.95):
    """Wilson score interval for a binomial proportion."""
    z = norm.ppf(0.5 + confidence_level / 2)
    p_hat = n_correct / n_total
    denom = 1 + z ** 2 / n_total
    centre = (p_hat + z ** 2 / (2 * n_total)) / denom
    margin = z * math.sqrt(
        (p_hat * (1 - p_hat) + z ** 2 / (4 * n_total)) / n_total
    ) / denom
    return max(0, centre - margin), min(1, centre + margin)

## Assignment 5.1: Run on the test set

Use your best configuration and run it to evaluate the test set

In [ ]:
log_test = eval(
    Task(
        dataset=TEST_SET,
        solver=# YOUR CODE HERE,
        scorer=# YOUR CODE HERE,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=len(TEST_SET),
    max_connections=1,
    timeout=120,
)[0]

n_test = # YOUR CODE HERE
n_correct_test = # YOUR CODE HERE

lo, hi = wilson_ci(n_correct_test, n_test)
print(f"Test accuracy : {n_correct_test / n_test:.1%}")
print(f"95% Wilson CI : [{lo:.1%}, {hi:.1%}]")
print(f"n = {n_test}")

## Assignment 5.2: Breakdown by subject and difficulty level

There aren't enough samples per category to draw firm conclusions — that's fine.
Just see if anything stands out.

Iterate over logs and produce **two separate tables**: one grouped
by subject, one by difficulty level. 

Your output should look like this:

**By subject:**

| Subject | Correct | Total | Acc |
|---|---:|---:|---:|
| Algebra | 1 | 10 | 10% |
| Precalculus | 3 | 6 | 50% |
| ... | | | |

**By level:**

| Level | Correct | Total | Acc |
|---|---:|---:|---:|
| 1 | 4 | 5 | 80% |
| 2 | 3 | 6 | 50% |
| ... | | | |

The actual numbers will appear in the cell below.

In [ ]:
subject_stats = defaultdict(lambda: [0, 0])
level_stats = defaultdict(lambda: [0, 0])

for sample in log_test.samples:
    sc = _first_score(sample)
    correct = 1 if sc.value == "C" else 0
    subj = sample.metadata.get("subject", "unknown")
    lvl = sample.metadata.get("level", 0)
    subject_stats[subj][0] += correct
    subject_stats[subj][1] += 1
    level_stats[lvl][0] += correct
    level_stats[lvl][1] += 1

print(f"{'Subject':<30s} {'Correct':>7s} {'Total':>5s} {'Acc':>6s}")
print("-" * 50)
for subj in sorted(subject_stats):
    c, t = subject_stats[subj]
    print(f"{subj:<30s} {c:>7d} {t:>5d} {c/t:>6.0%}")

print()
print(f"{'Level':<30s} {'Correct':>7s} {'Total':>5s} {'Acc':>6s}")
print("-" * 50)
for lvl in sorted(level_stats):
    c, t = level_stats[lvl]
    print(f"Level {lvl:<24d} {c:>7d} {t:>5d} {c/t:>6.0%}")

**Your results:**

Fill in the tables once you've run the cell above.

**By subject:**

| Subject | Correct | Total | Acc |
|---|---:|---:|---:|
| Algebra | | | |
| Precalculus | | | |
| ... | | | |

**By difficulty level:**

| Level | Correct | Total | Acc |
|---|---:|---:|---:|
| 1 | | | |
| 2 | | | |
| ... | | | |

## Final comparison: dev vs test

In [ ]:
print(f"{'Configuration':<30s} {'Dev Acc':>8s}  {'Test Acc':>8s}")
print("=" * 50)
for name, log in DEV_RUNS:
    acc = get_acc(log)
    print(f"{name:<30s} {acc:>8.0%}  {'--':>8s}")
print(f"{'best agent (TEST)':<30s} {'--':>8s}  {n_correct_test / n_test:>8.1%}")
print(f"\n95% CI on test: [{lo:.1%}, {hi:.1%}]")

---

### 💭 Questions for reflection

1. How does the test accuracy compare to the dev accuracy? If there's a gap,
   what might explain it?
2. Which subjects and difficulty levels does the agent handle best? Worst?
3. Look at the confidence interval. Is it narrow enough to be useful, or would
   you want more test samples?
4. If you were to improve this agent further, where would you focus?

**Your answer:**

## Bonus assignment: Error Analysis

Pick 5-10 test samples that the agent got wrong and classify each failure. Here's one
possible taxonomy — feel free to use your own:

- **Tool misuse:** Called the wrong tool or with wrong arguments
- **Reasoning error:** Correct tool use but flawed multi-step reasoning
- **Format error:** Correct answer but wrong format in `submit()`
- **Inherent difficulty:** Problem requires reasoning beyond the tool set
- **Grading error:** The agent was actually right but the scorer got it wrong
- **Other:** Anything that doesn't fit the above

This kind of qualitative error analysis is essential in practice — aggregate metrics
tell you *how much* the agent struggles, but error analysis tells you *why*.

If you'd like to explore the logs in code, use the cell below — otherwise just read
them directly and delete it.

In [ ]:
# YOUR CODE HERE

---

### 💭 Questions for reflection

1. What was the most common failure mode?
2. Which failure modes could be fixed with better prompting vs. better tools
   vs. a more capable model?
3. Did you find any grading errors? What does that imply about using model-graded scoring?

**Your answer:**

## Bonus assignment 2: Automated Error Analysis with Inspect Scout

**Inspect Scout** is a utility designed to analyze Large Language Model (LLM) execution transcripts. 
The core data pipeline is structured as follows:

Transcript->Scanner->Result

- **Transcript**: It can be an `EvalLog` generated by Inspect AI.
- **Scanner**: A function that inspects specific elements within the Transcript.
It can use LLMs or regular expressions. You can also create a custom Scanner.
- **Result**: The output of the scanning process. You can review
results in the `Scans` section using the `scout view` command.

Inspect Scout is similar to Inspect AI and acts as the next step in your data pipeline. 
It allows you to measure metrics that can be used to detect evaluation awareness and other features.

In this tutorial, we will use Inspect Scout to group transcripts by their failure reasons
(part 1) and calculate tool use metrics (part 2).
To complete this task, you'll need an `.eval` file from this (4th) tutorial with plenty of tasks
(the more, the better).

For more info about Inspect Scout, see [docs](https://meridianlabs-ai.github.io/inspect_scout/)

In [ ]:
# inspect-scout needs inspect-ai>=0.3.239 (pinned in the setup cell), so this won't upgrade it mid-session.
%pip install -q inspect-scout

In [ ]:
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from inspect_scout import Scanner, Transcript, Result
from inspect_scout import llm_scanner, scanner, scan, transcripts_from, scan_results_df
from inspect_ai.event import ToolEvent
from pandas.api.types import is_integer_dtype

### Bonus Assignment 2, part 1: Failure Analysis

In the first part, you will implement a simple scanner to determine the root cause of notebook execution failures.

**Note:** It is recommended to use a powerful LLM for this task. You may also want to limit the total number of processed samples.

In [ ]:
# insert path to your transcript (.eval file) here
TRANSCRIPT_PATH =  # YOUR CODE HERE

Here you can enter a prompt for the LLM classifying the reasons for failure using the classification from the previous task (you can also use your own classification here).

`llm_scanner` is a scanner that checks a transcript using an LLM with a custom `question`. See the sample below.

```python
@scanner(messages="all")
def scanner_leaked_secrets() -> Scanner[Transcript]:
    return llm_scanner(
        question='Do these messages have leaked secrets/configs/passwords? If so, answer "yes", else answer "no". Answer with only one word.',
        answer="string",
        model='ollama/qwen2.5',
    )
```

In [ ]:
@scanner(messages="all")
def scanner_failure_classifier() -> Scanner[Transcript]:
    return llm_scanner(
        question= # YOUR CODE HERE
        answer="string",
        model=  # YOUR CODE HERE
    )

Run a scan (it may take a while):

In [ ]:
scan_result = scan(scanners=[scanner_failure_classifier()], transcripts=transcripts_from(TRANSCRIPT_PATH))

Here you can process data.

In [ ]:
df = scan_results_df(scan_result.location).scanners['scanner_failure_classifier']

In [ ]:
# 1. Remove all columns except ['transcript_task_id', 'transcript_success', 'value'] in df_short.
# 2. Copy the result to df_short_failed.
# 3. Filter df_short_failed by transcript status.

df_short = # YOUR CODE HERE

# YOUR CODE HERE

df_short_failed = # YOUR CODE HERE

In [ ]:
# TESTS

assert len(df) == len(df_short)
assert len(df_short.columns) == 3, f"Expected 3 columns, got {len(df_short.columns)}"
_expected_cols = {'transcript_task_id', 'transcript_success', 'value'}
assert set(df_short.columns) == _expected_cols, f"Columns mismatch. Expected {_expected_cols}, got {set(df_short.columns)}"

assert len(df_short_failed.columns) == 3, f"Expected 3 columns, got {len(df_short_failed.columns)}"
assert set(df_short_failed.columns) == _expected_cols, f"Columns mismatch. Expected {_expected_cols}, got {set(df_short_failed.columns)}"
assert len(df_short_failed) > 0, "df_short_failed is empty, but expected non empty"
assert len(df_short_failed) < len(df_short), "df_short_failed is >= df_short, expected <"

assert df_short['value'].notna().all(), "Found missing values in df_short['value']"
assert df_short_failed['value'].notna().all(), "Found missing values in df_short_failed['value']"
assert (df_short_failed['transcript_success'] == False).all(), f"Found non-False values: {df_short_failed['transcript_success'].unique()}"

You can investigate `df`.

For example, check diagram below.

In [ ]:
# Barchart of failures modes
column_labels = list(df_short_failed.value)
counts = Counter(column_labels)
labels = list(counts.keys())
sizes = list(counts.values())

fig, ax = plt.subplots()
ax.bar(labels, sizes, color=plt.cm.tab10(range(len(labels))))
ax.set_xlabel('Label')
ax.set_ylabel('Count')
ax.set_title('Label Frequency')
ax.set_xticks(list(range(len(labels))))
ax.set_xticklabels(labels, rotation=45, fontsize=8)
plt.show()

### Bonus Assignment 2, part 2: Tool Call Metrics

In the second part, you will build a custom Scanner to analyze tool call statistics. 
You will track tool call number and errors.

In [ ]:
@scanner(events=['tool'])
def scanner_tool_calls_analyzer() -> Scanner[Transcript]:
    """Tool calls analyzer. 

    It should count 2 things,
    - number of tool calls with error
    - total number of tool calls 
    """
    async def scan(transcript: Transcript) -> Result:
        #  YOUR CODE HERE
       
        for evnt in transcript.events:
            evnt: ToolEvent
            if 'error' in str(evnt.result).lower() or evnt.error is not None:
                pass  # YOUR CODE HERE
        
        return Result(
            value=count_errors,
            metadata={
                "tool_calls": -1 # YOUR CODE HERE
            },
        )
    return scan

In [ ]:
scan_result_2 = scan(scanners=[scanner_tool_calls_analyzer()], 
                     transcripts=transcripts_from(TRANSCRIPT_PATH))

In [ ]:
df_2 = scan_results_df(scan_result_2.location).scanners['scanner_tool_calls_analyzer']

Now you have basic data about tool use (number of tool calls and number of tool calls with error result) in df_2.

We can plot a hist of it. 

To do it, let's create `short_df_2` dataframe with columns `tool_calls` and `error_tool_calls`.

In [ ]:
from json import loads

# YOUR CODE HERE

In [ ]:
# TESTS

assert len(short_df_2.columns) > 0
assert len(short_df_2.columns) == 2
assert set(short_df_2.columns) == {"error_tool_calls", "tool_calls"}
assert len(short_df_2) > 0
assert is_integer_dtype(short_df_2['error_tool_calls'].dtype)
assert is_integer_dtype(short_df_2['tool_calls'].dtype)

Plot final histogram of tool calls.

In [ ]:
short_df_2['error_tool_calls'].hist(
    figsize=(8, 4), 
    alpha=0.5, 
    label='Error tool calls', 
    bins=30,
    color='red'
)
short_df_2['tool_calls'].hist(
    alpha=0.5, 
    label='Total tool calls', 
    bins=30,
    color='blue'
)

plt.title(f"Tool calls")
plt.legend()
plt.show()


### Bonus Assignment 2, extra part: Evaluation Awareness and Inspect Scout.

[The docs](https://meridianlabs-ai.github.io/inspect_scout/) notes that Inspect Scout
can detect **"evaluation awareness"**. 

"Evaluation awareness" is a model’s ability to distinguish evaluation settings
from real-world deployment settings. A narrower and easier-to-detect case is
**"verbalized evaluation awareness"**: when the model explicitly expresses 
in its reasoning or final response that it may be 
evaluated, tested, benchmarked, or placed in an artificial evaluation setting.

Researchers investigated this behavior in the arXiv preprint 
["Large Language Models Often Know When They Are Being Evaluated"](https://arxiv.org/abs/2505.23836).

For this free-form task, try to **partially replicate** this article, using your available resources
(or any other "eval awareness" article).
You may use any model and dataset to detect this behavior.

Did you get the same result? **Describe** your replication process and results.